# LogSeer Training

Trains any combination of neural network and sklearn models (xgb, svm, rf) to classify server logs as success or failure.

Run cells from top to bottom. Colab-only cells are marked — skip them if running elsewhere.

**Setup**: Place log data under `DATA_DIR` with this structure:

```
DATA_DIR/
├── error/
│   ├── 1/
│   │   ├── log1.txt
│   │   └── log2.txt
│   └── 2/
│       └── log1.txt
└── success/
    ├── 1/
    │   └── log1.txt
    └── 2/
        └── log1.txt
```

Each subdirectory under `error/` and `success/` is one operation run. Adjust the Configuration cell as needed — key knobs are `SUCCESS_LOG_RATIO` (class balance), `NN_ERROR_WEIGHT` / `SKLEARN_WEIGHT` (error-class emphasis), `SKLEARN_MODELS` (e.g. `'xgb,svm'`), and `TEST_NN` (set to `False` to skip the NN entirely).

**Output**: saves `logseer.keras`, `tokenizer.pickle`, and one `.pkl` per sklearn model to the repo root. Run the last cell to copy them to Google Drive.

In [ ]:
# Setup
import os, sys

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

if not os.path.exists('logseer'):
    os.system('git clone https://github.com/masahiko-shibata/logseer.git')
    os.chdir('logseer')

sys.path.insert(0, '.')

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow as tf
from logseer import *

if not tf.config.list_physical_devices('GPU'):
    print("⚠️ WARNING: No GPU detected! Will run on CPU. Nural network training will be very slow. If you are on Colab, consider changing Runtime > Change runtime type > Hardware accelerator.")
else:
    print("✅ GPU is available:", tf.config.list_physical_devices('GPU'))

**Colab only** — Run below if you want to load training data from a zip file on Google Drive. Default looks for `Colab Notebooks/logseer/data/data.zip` in your My Drive. Zip the `data/` folder itself so the zip extracts to a `data/` folder — do not zip the contents directly. Otherwise copy data to `DATA_DIR` manually.

In [ ]:
# Data copy from Google Drive
from google.colab import drive
import shutil, zipfile
drive.mount('/content/drive')

!rm -rf logs data.zip 2>/dev/null
shutil.copy('/content/drive/MyDrive/Colab Notebooks/logseer/data/data.zip', 'data.zip')
with zipfile.ZipFile('data.zip', 'r') as z:
    z.extractall('.')
print(os.listdir('./'))

In [ ]:
# Configuration

# Paths
DATA_DIR            = 'data'
MODEL_SAVE_PATH     = 'logseer.keras'
TOKENIZER_PATH      = 'tokenizer.pickle'

# Data
NUM_CHAR            = 3000
TO_ID               = 6000
VALIDATION_SPLIT      = 0.1
VALIDATE_ON_TEST_DATA = False
SUCCESS_LOG_RATIO     = 99

# Model
MODEL_NAME          = 'LogCNNv2'
EMBEDDING_LAYER     = 'vanilla'
EMBEDDING_DIM       = 32
MAX_NB_WORDS        = 24000
MAX_SEQUENCE_LENGTH = 26000

# Training
EPOCHS              = 60
BATCH_SIZE          = 32
LEARNING_RATE       = 0.0003
REPETITION          = 100
RETRAIN             = False
NN_ERROR_WEIGHT     = 2
LABEL_SMOOTHING     = 0.1

# Early stopping
USE_EARLY_STOPPING   = True
PATIENCE             = 15
RESTORE_BEST_WEIGHTS = False
ES_START_FROM_EPOCH  = 10
MONITOR              = 'val_f1'
MODE                 = 'max'

# Checkpoint: 'multi_metric' | 'best_f1' | 'standard'
CHECKPOINT_TYPE      = 'best_f1'
START_FROM_EPOCH     = 0
MAX_LOSS            = 0.7

# Sklearn models to train alongside the NN: 'xgb', 'svm', 'rf' (comma-separated string or list)
SKLEARN_MODELS      = 'xgb'
SKLEARN_WEIGHT      = 6

In [ ]:
# Training loop
tester = run_training(
    DATA_DIR,
    max_nb_words=MAX_NB_WORDS,
    max_sequence_length=MAX_SEQUENCE_LENGTH,
    embedding_dim=EMBEDDING_DIM,
    validation_split=VALIDATION_SPLIT,
    validate_on_test_data=VALIDATE_ON_TEST_DATA,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    model_save_path=MODEL_SAVE_PATH,
    tokenizer_path=TOKENIZER_PATH,
    embedding_layer_type=EMBEDDING_LAYER,
    model_name=MODEL_NAME,
    repetition=REPETITION,
    nn_error_weight=NN_ERROR_WEIGHT,
    label_smoothing=LABEL_SMOOTHING,
    learning_rate=LEARNING_RATE,
    max_loss=MAX_LOSS,
    retrain=RETRAIN,
    numchar=NUM_CHAR,
    toid=TO_ID,
    checkpoint_type=CHECKPOINT_TYPE,
    start_from_epoch=START_FROM_EPOCH,
    es_start_from_epoch=ES_START_FROM_EPOCH,
    use_early_stopping=USE_EARLY_STOPPING,
    patience=PATIENCE,
    monitor=MONITOR,
    mode=MODE,
    restore_best_weights=RESTORE_BEST_WEIGHTS,
    sklearn_models=SKLEARN_MODELS,
    sklearn_weight=SKLEARN_WEIGHT,
)

**Colab only** — saves trained model files to `Colab Notebooks/logseer/models' folder in your My Drive.

In [ ]:
# Copy model files to Google Drive
import shutil
from google.colab import drive
drive.mount('/content/drive')
DRIVE_MODEL_DIR = '/content/drive/MyDrive/Colab Notebooks/logseer/models'
os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)
shutil.copy(MODEL_SAVE_PATH, DRIVE_MODEL_DIR + '/' + MODEL_SAVE_PATH)
shutil.copy(TOKENIZER_PATH,  DRIVE_MODEL_DIR + '/' + TOKENIZER_PATH)
for m in [m.strip() for m in SKLEARN_MODELS.replace(',', ' ').split() if m.strip()]:
    shutil.copy(f'{m}.pkl', DRIVE_MODEL_DIR + f'/{m}.pkl')
print('Models saved to', DRIVE_MODEL_DIR)